In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/split_part_4.csv"

df = pd.read_csv(CSV_PATH, low_memory=False)

print("Dataset loaded!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded!
Rows: 3257967
Columns: 24


In [ ]:
# just checking what the data looks like
pd.set_option('display.max_columns', None)
df.head()

,Invoice/Item Number,Date,Store Number,Store Name,Address,City,Zip Code,Store Location,County Number,County,Category,Category Name,Vendor Number,Vendor Name,Item Number,Item Description,Pack,Bottle Volume (ml),State Bottle Cost,State Bottle Retail,Bottles Sold,Sale (Dollars),Volume Sold (Liters),Volume Sold (Gallons)
0,INV-79454000032,02/09/2025,6338,TOBACCO HUT PLUS US GAS / DES MOINES,3000 SW 9TH STREET,DES MOINES,50315.0,POINT (-93.62614 41.55811),NaN,POLK,1081600,WHISKEY LIQUEUR,421,SAZERAC COMPANY INC,64870,FIREBALL CINNAMON WHISKEY,48,100,$1.00,$1.50,480,$720.00,48.00,12.68
1,INV-79461000055,02/10/2025,6242,WALL TO WALL WINE AND SPIRITS / WEST DES MOINES,375 SOUTH JORDAN CREEK PARKWAY,WEST DES MOINES,50266.0,POINT (-93.8106 41.56318),NaN,DALLAS,1701100,TEMPORARY & SPECIALTY PACKAGES,460,SHAW-ROSS INTERNATIONAL,46688,PAPAS PILAR BLONDE RUM,6,750,$18.30,$27.45,12,$329.40,9.00,2.37
2,INV-79485800002,02/10/2025,5708,BROTHERS MARKET / WILLIAMSBURG,103 W. WALNUT,WILLIAMSBURG,52361.0,POINT (-92.00797 41.66282),NaN,IOWA,1012100,CANADIAN WHISKIES,259,HEAVEN HILL BRANDS,11788,BLACK VELVET,6,"1,750",$11.50,$17.25,18,$310.50,31.50,8.32
3,INV-79477100016,02/10/2025,5736,218 FUEL EXPRESS,68 MONROE STREET,FLOYD,50435.0,POINT (-92.73781 43.12232),NaN,FLOYD,1031200,AMERICAN FLAVORED VODKA,380,PHILLIPS BEVERAGE,42079,UV CAKE,12,"1,000",$7.50,$11.25,2,$22.50,2.00,0.52
4,INV-79470900015,02/10/2025,5436,SMOKIN' JOE'S #8 TOBACCO AND LIQUOR OUTLET,902 W. KIMBERLY RD STE 55-56,DAVENPORT,52806.0,POINT (-90.58492 41.56041),NaN,SCOTT,1051100,AMERICAN BRANDIES,421,SAZERAC COMPANY INC,53218,PAUL MASSON GRANDE AMBER BRANDY VS,6,"1,750",$10.00,$15.00,6,$90.00,10.50,2.77


In [ ]:
# checking all column names and their data types
df.dtypes

,0
Invoice/Item Number,object
Date,object
Store Number,int64
Store Name,object
Address,object
City,object
Zip Code,float64
Store Location,object
County Number,float64
County,object


In [ ]:
# total null count per column
null_counts = df.isnull().sum()
# converting to percentage
null_percent = (null_counts / len(df)) * 100
# putting it together in one table
null_summary = pd.DataFrame({
    'null_count': null_counts,
    'null_percent': null_percent.round(2)
})
# only showing columns that actually have nulls
null_summary = null_summary[null_summary['null_count'] > 0]
null_summary.sort_values('null_percent', ascending=False)


,null_count,null_percent
County Number,3237190,99.36
Store Location,22001,0.68
City,1897,0.06
Address,1897,0.06
Zip Code,1897,0.06
County,1897,0.06


In [ ]:
# if any of these are null, that row is not useful

critical_columns = [
    "Address", "City", "Zip Code",
    "Volume Sold (Gallons)", "Volume Sold (Liters)",
    "Sale (Dollars)", "Bottles Sold",
    "State Bottle Retail", "Bottle Volume (ml)",
    "Pack", "Item Number", "Vendor Name",
    "Vendor Number", "Category Name", "Category"
]
# checking how many rows will be dropped if we remove nulls in these columns
critical_cols_in_df = [c for c in critical_columns if c in df.columns]
rows_with_nulls = df[critical_cols_in_df].isnull().any(axis=1).sum()

print("Rows that will be dropped due to missing critical values:", rows_with_nulls)
print("That is", round(rows_with_nulls / len(df) * 100, 2), "% of the data")


Rows that will be dropped due to missing critical values: 1897
That is 0.06 % of the data


In [ ]:
# store location had a lot of nulls, let check exactly how many
print("Total rows:", len(df))
print("Store Location - null count:", df["Store Location"].isnull().sum())
print("Store Location - null %:", round(df["Store Location"].isnull().mean() * 100, 1), "%")


Total rows: 3257967
Store Location - null count: 22001
Store Location - null %: 0.7 %


In [ ]:
# store location is redundant - safe to drop
print("We already have these location columns:")
print(df[["Address", "City", "Zip Code"]].head())


We already have these location columns:
                          Address             City  Zip Code
0              3000 SW 9TH STREET       DES MOINES   50315.0
1  375 SOUTH JORDAN CREEK PARKWAY  WEST DES MOINES   50266.0
2                   103 W. WALNUT     WILLIAMSBURG   52361.0
3                68 MONROE STREET            FLOYD   50435.0
4    902 W. KIMBERLY RD STE 55-56        DAVENPORT   52806.0


In [ ]:
# finding columns that have spaces or slashes in their names
problem_columns = [col for col in df.columns if " " in col or "/" in col]

print("Columns with naming issues:")
for col in problem_columns:
    print(" -", col)


Columns with naming issues:
 - Invoice/Item Number
 - Store Number
 - Store Name
 - Zip Code
 - Store Location
 - County Number
 - Category Name
 - Vendor Number
 - Vendor Name
 - Item Number
 - Item Description
 - Bottle Volume (ml)
 - State Bottle Cost
 - State Bottle Retail
 - Bottles Sold
 - Sale (Dollars)
 - Volume Sold (Liters)
 - Volume Sold (Gallons)


In [ ]:
# Invoice/Item Number is the biggest issue because of the slash
# we will rename it to Invoice_Number
print("Invoice/Item Number sample values:")
print(df["Invoice/Item Number"].head())

Invoice/Item Number sample values:
0    INV-79454000032
1    INV-79461000055
2    INV-79485800002
3    INV-79477100016
4    INV-79470900015
Name: Invoice/Item Number, dtype: object


In [ ]:
# from the dataset schema we can see these ID columns are Text type
# but they should be integers for proper analysis
id_columns = ["Store Number", "County Number", "Category", "Vendor Number", "Item Number"]

print("ID columns that are stored as Text but should be integers:")
for col in id_columns:
    if col in df.columns:
        print(f"\n  {col}")
        print("    dtype  :", df[col].dtype)
        print("    sample :", df[col].dropna().head(3).tolist())


ID columns that are stored as Text but should be integers:

  Store Number
    dtype  : int64
    sample : [6338, 6242, 5708]

  County Number
    dtype  : float64
    sample : [57.0, 56.0, 77.0]

  Category
    dtype  : int64
    sample : [1081600, 1701100, 1012100]

  Vendor Number
    dtype  : int64
    sample : [421, 460, 259]

  Item Number
    dtype  : int64
    sample : [64870, 46688, 11788]


In [ ]:
# checking if these columns have any non-numeric values that would break casting
print("Checking for non-numeric values in ID columns:")
for col in id_columns:
    if col in df.columns:
        non_numeric = pd.to_numeric(df[col], errors='coerce').isna().sum()
        print(f"  {col} -> non-numeric values: {non_numeric}")


Checking for non-numeric values in ID columns:
  Store Number -> non-numeric values: 0
  County Number -> non-numeric values: 3237190
  Category -> non-numeric values: 0
  Vendor Number -> non-numeric values: 0
  Item Number -> non-numeric values: 0


In [ ]:
# checking how the date column looks after loading
print("Date column dtype:", df["Date"].dtype)
print("Sample values:")
print(df["Date"].dropna().head(5).tolist())


Date column dtype: object
Sample values:
['02/09/2025', '02/10/2025', '02/10/2025', '02/10/2025', '02/10/2025']


In [ ]:
# checking how many dates cannot be parsed - those rows will be dropped
bad_dates = pd.to_datetime(df["Date"], errors='coerce').isna().sum()
print("Rows where date cannot be parsed:", bad_dates)


Rows where date cannot be parsed: 0


In [ ]:
# checking the issue - county name exists but number is missing
has_county_name  = df["County"].notna()
no_county_number = df["County Number"].isna()

mismatch = (has_county_name & no_county_number).sum()
print("Rows where County name exists but County Number is missing:", mismatch)


Rows where County name exists but County Number is missing: 3235293


In [ ]:
# also checking if county names have inconsistent casing or spaces
print("Sample county values from the data:")
print(df["County"].dropna().unique()[:20])


Sample county values from the data:
['POLK' 'DALLAS' 'IOWA' 'FLOYD' 'SCOTT' 'LINN' 'JOHNSON' 'POTTAWATTAMIE'
 'DUBUQUE' 'CEDAR' 'DES MOINES' 'JASPER' 'WEBSTER' 'HUMBOLDT' 'DELAWARE'
 'WASHINGTON' 'JONES' 'SAC' 'WARREN' 'WINNESHIEK']


In [ ]:
# from the dataset description we know:
# Volume Sold (Liters) = Bottle Volume (ml) x Bottles Sold / 1000
# so if liters is null but the other two exist, we can calculate it

liters_null       = df["Volume Sold (Liters)"].isna()
ml_available      = df["Bottle Volume (ml)"].notna()
bottles_available = df["Bottles Sold"].notna()

can_calculate_liters = (liters_null & ml_available & bottles_available).sum()
print("Rows where Volume (Liters) is null but can be calculated:", can_calculate_liters)


Rows where Volume (Liters) is null but can be calculated: 0


In [ ]:

# Volume Sold (Gallons) = Bottle Volume (ml) x Bottles Sold / 3785.411784
# but simpler to convert from liters: Liters x 0.264172

gallons_null     = df["Volume Sold (Gallons)"].isna()
liters_available = df["Volume Sold (Liters)"].notna()

can_calculate_gallons = (gallons_null & liters_available).sum()
print("Rows where Volume (Gallons) is null but can be calculated:", can_calculate_gallons)


Rows where Volume (Gallons) is null but can be calculated: 0


In [ ]:
# printing all column names to confirm no profit column exists
print("All columns in raw dataset:")
for col in df.columns:
    print(" -", col)


All columns in raw dataset:
 - Invoice/Item Number
 - Date
 - Store Number
 - Store Name
 - Address
 - City
 - Zip Code
 - Store Location
 - County Number
 - County
 - Category
 - Category Name
 - Vendor Number
 - Vendor Name
 - Item Number
 - Item Description
 - Pack
 - Bottle Volume (ml)
 - State Bottle Cost
 - State Bottle Retail
 - Bottles Sold
 - Sale (Dollars)
 - Volume Sold (Liters)
 - Volume Sold (Gallons)


In [ ]:
# profit per bottle is not there but we have what we need to calculate it
# profit_per_bottle = State Bottle Retail - State Bottle Cost
print("State Bottle Cost sample  :", df["State Bottle Cost"].dropna().head(3).tolist())
print("State Bottle Retail sample:", df["State Bottle Retail"].dropna().head(3).tolist())


State Bottle Cost sample  : ['$1.00', '$18.30', '$11.50']
State Bottle Retail sample: ['$1.50', '$27.45', '$17.25']


In [ ]:
# also need to check if Bottles Sold ever has 0 values
# because we will divide Sale (Dollars) by Bottles Sold for revenue per bottle
zero_bottles = (df["Bottles Sold"] == 0).sum()
print("Rows where Bottles Sold = 0:", zero_bottles)
print("We need to handle this when calculating Revenue per Bottle to avoid divide by zero")


Rows where Bottles Sold = 0: 0
We need to handle this when calculating Revenue per Bottle to avoid divide by zero
